### Target Definition and Net Flow Metric

In [0]:
# ============================================================
# Task 1 — Define Targets and Net Flow Metric
# ============================================================

from pyspark.sql import functions as F

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
SILVER_AGG_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/station_hour_flow"
SILVER_STATION = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/station_id"

DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

# ------------------------------------------------------------
# 1) Load station-hour flow and station metadata
# ------------------------------------------------------------
df_flow = spark.read.parquet(SILVER_AGG_DIR)
df_station = spark.read.parquet(SILVER_STATION)

# ------------------------------------------------------------
# 2) Filter stations located in downtown Toronto
# ------------------------------------------------------------
df_base = (
    df_flow
    .join(df_station.select("station_id", "name", "lat", "lon"), on="station_id", how="left")
    .filter(
        (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
        (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
    )
)

# ------------------------------------------------------------
# 3) Define core targets and derived station demand metrics
# ------------------------------------------------------------
df_targets = (
    df_base
    .withColumn("target_departures", F.col("departures"))
    .withColumn("target_arrivals", F.col("arrivals"))
    .withColumn("net_flow", F.col("arrivals") - F.col("departures"))
    .withColumn("total_demand", F.col("arrivals") + F.col("departures"))
)

# ------------------------------------------------------------
# 4) Basic validation
# ------------------------------------------------------------
print("Rows in downtown dataset:", f"{df_targets.count():,}")
print("Distinct downtown stations:", df_targets.select("station_id").distinct().count())

display(
    df_targets.select(
        "station_id", "year", "month", "day", "hour",
        "departures", "arrivals",
        "target_departures", "target_arrivals",
        "net_flow", "total_demand"
    ).limit(10)
)

# ------------------------------------------------------------
# 5) Summary stats for documentation
# ------------------------------------------------------------
summary = df_targets.select(
    F.mean("target_departures").alias("avg_departures"),
    F.mean("target_arrivals").alias("avg_arrivals"),
    F.mean("net_flow").alias("avg_net_flow"),
    F.mean("total_demand").alias("avg_total_demand"),
    F.expr("percentile_approx(target_departures, 0.50)").alias("p50_departures"),
    F.expr("percentile_approx(target_arrivals, 0.50)").alias("p50_arrivals"),
    F.expr("percentile_approx(total_demand, 0.50)").alias("p50_total_demand")
)

display(summary)

The station-hour dataset contains over 2.4 million observations across 228 downtown stations.  
On average, stations experience approximately **2.76 departures and 2.78 arrivals per hour**, resulting in an almost neutral **average net flow close to zero**.

However, the **median total demand per station-hour is only 4 trips**, indicating that most stations operate under relatively low activity levels while imbalance events occur intermittently.  

This highlights the importance of modeling **temporal patterns and contextual factors** (such as time of day, historical usage, weather, and nearby events) to accurately predict fluctuations in station demand and potential rebalancing needs.

### Engineering Predictive Features

In [0]:
# ============================================================
# Task 2 — Engineer Predictive Features
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------
# 1) Base dataset (targets already defined)
# ------------------------------------------------------------
df_feat = df_targets

# ------------------------------------------------------------
# 2) Calendar features
# ------------------------------------------------------------
df_feat = (
    df_feat
    .withColumn("date", F.make_date("year", "month", "day"))
    .withColumn("dow_num", F.dayofweek("date"))  # 1=Sun ... 7=Sat
    .withColumn("is_weekend", F.col("dow_num").isin([1, 7]).cast("int"))
)

# ------------------------------------------------------------
# 3) Window definition for station time series
# ------------------------------------------------------------
w = Window.partitionBy("station_id").orderBy(F.col("date"), F.col("hour"))

# ------------------------------------------------------------
# 4) Lag features (historical departures)
# ------------------------------------------------------------
df_feat = (
    df_feat
    .withColumn("lag1_dep", F.lag("target_departures", 1).over(w))
    .withColumn("lag2_dep", F.lag("target_departures", 2).over(w))
    .withColumn("lag3_dep", F.lag("target_departures", 3).over(w))
    .withColumn("lag24_dep", F.lag("target_departures", 24).over(w))
    .withColumn("lag48_dep", F.lag("target_departures", 48).over(w))
    .withColumn("lag168_dep", F.lag("target_departures", 168).over(w))
)

# ------------------------------------------------------------
# 5) Rolling statistics (history-only, no leakage)
# ------------------------------------------------------------
w_roll3 = Window.partitionBy("station_id").orderBy(F.col("date"), F.col("hour")).rowsBetween(-3, -1)
w_roll24 = Window.partitionBy("station_id").orderBy(F.col("date"), F.col("hour")).rowsBetween(-24, -1)

df_feat = (
    df_feat
    .withColumn("roll_mean_3h", F.avg("target_departures").over(w_roll3))
    .withColumn("roll_mean_24h", F.avg("target_departures").over(w_roll24))
    .withColumn("roll_std_24h", F.stddev("target_departures").over(w_roll24))
)

# ------------------------------------------------------------
# 6) Remove rows without sufficient history
# ------------------------------------------------------------
df_feat_model = df_feat.dropna()

print("Rows after feature engineering:", f"{df_feat_model.count():,}")

display(
    df_feat_model.select(
        "station_id", "year", "month", "day", "hour",
        "target_departures",
        "lag1_dep", "lag24_dep", "lag168_dep",
        "roll_mean_3h", "roll_mean_24h", "roll_std_24h",
        "dow_num", "is_weekend"
    ).limit(10)
)

The dataset retains approximately **2.36 million valid station-hour observations**, indicating that most stations have sufficient historical coverage to support temporal feature engineering.

Lag variables capture immediate demand history (1–3 hours) as well as daily and weekly behavioral cycles (24 and 168 hours). Rolling statistics summarize short-term demand trends and variability, providing the model with contextual information about recent station activity.

These engineered features introduce the temporal dependencies required for forecasting bike demand at the station-hour level.

### Preparing the Model-Ready Dataset

In [0]:
# ============================================================
# Task 3 — Prepare Model-Ready Dataset
# ============================================================

# ------------------------------------------------------------
# 1) Define target variable
# ------------------------------------------------------------
TARGET = "target_departures"

# ------------------------------------------------------------
# 2) Define final feature set
# ------------------------------------------------------------
model_features = [
    "station_id",
    "month",
    "hour",
    "dow_num",
    "is_weekend",
    "lag1_dep",
    "lag2_dep",
    "lag3_dep",
    "lag24_dep",
    "lag48_dep",
    "lag168_dep",
    "roll_mean_3h",
    "roll_mean_24h",
    "roll_std_24h"
]

# ------------------------------------------------------------
# 3) Select model-ready columns
# ------------------------------------------------------------
df_model = df_feat_model.select(
    "station_id", "year", "month", "day", "hour",
    "dow_num", "is_weekend",
    "lag1_dep", "lag2_dep", "lag3_dep",
    "lag24_dep", "lag48_dep", "lag168_dep",
    "roll_mean_3h", "roll_mean_24h", "roll_std_24h",
    TARGET
)

print("Model-ready dataset rows:", f"{df_model.count():,}")
print("Number of model features:", len(model_features))

display(df_model.limit(10))

# ------------------------------------------------------------
# 4) Save model-ready dataset
# ------------------------------------------------------------
MODEL_READY_PATH = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/downtown_dep_features_v1"

(
    df_model
    .write
    .mode("overwrite")
    .parquet(MODEL_READY_PATH)
)

print("Model-ready dataset saved to:", MODEL_READY_PATH)